In [3]:
import duckdb

con = duckdb.connect()

all_pattern = "data/processed/enriched_sales_all/store_id=*/*.parquet"

In [4]:
# Basic summary
con.execute(f"""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT store_id) AS stores,
    COUNT(DISTINCT item_id) AS items,
    MIN(date) AS min_date,
    MAX(date) AS max_date,
    SUM(units_sold) AS total_units
FROM read_parquet('{all_pattern}')
""").fetchdf()

,rows,stores,items,min_date,max_date,total_units
0,59181090,10,3049,2011-01-29,2016-05-22,66927173.0


In [5]:
con.execute(f"""
SELECT
    store_id,
    SUM(units_sold) AS units_sold,
    AVG(units_sold) AS avg_daily_item_sales
FROM read_parquet('{all_pattern}')
GROUP BY store_id
ORDER BY units_sold DESC
""").fetchdf()

,store_id,units_sold,avg_daily_item_sales
0,CA_3,11363540.0,1.920130
1,CA_1,7832248.0,1.323438
2,TX_2,7329642.0,1.238511
3,WI_2,6697988.0,1.131778
4,WI_3,6542557.0,1.105515
5,TX_3,6205940.0,1.048636
6,CA_2,5818395.0,0.983151
7,TX_1,5692823.0,0.961933
8,WI_1,5261506.0,0.889052
9,CA_4,4182534.0,0.706735


In [6]:
con.execute(f"""
SELECT
    cat_id,
    SUM(units_sold) AS units_sold
FROM read_parquet('{all_pattern}')
GROUP BY cat_id
ORDER BY units_sold DESC
""").fetchdf()

,cat_id,units_sold
0,FOODS,45922427.0
1,HOUSEHOLD,14764090.0
2,HOBBIES,6240656.0


In [7]:
con.execute(f"""
SELECT
    is_available,
    COUNT(*) AS rows,
    SUM(units_sold) AS units_sold
FROM read_parquet('{all_pattern}')
GROUP BY is_available
""").fetchdf()

,is_available,rows,units_sold
0,0,12299413,0.0
1,1,46881677,66927173.0


In [8]:
import duckdb

con = duckdb.connect()

all_pattern = "data/processed/enriched_sales_all/store_id=*/*.parquet"

In [9]:
test_pattern = (
    "data/processed/enriched_sales_all/"
    "store_id=CA_1/*.parquet"
)

In [10]:
con.execute(f"""
COPY (
    SELECT
        *,

        LAG(units_sold, 1) OVER (
            PARTITION BY store_id, item_id
            ORDER BY date
        ) AS lag_1,

        LAG(units_sold, 7) OVER (
            PARTITION BY store_id, item_id
            ORDER BY date
        ) AS lag_7,

        LAG(units_sold, 28) OVER (
            PARTITION BY store_id, item_id
            ORDER BY date
        ) AS lag_28,

        AVG(units_sold) OVER (
            PARTITION BY store_id, item_id
            ORDER BY date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS rolling_mean_7,

        AVG(units_sold) OVER (
            PARTITION BY store_id, item_id
            ORDER BY date
            ROWS BETWEEN 28 PRECEDING AND 1 PRECEDING
        ) AS rolling_mean_28

    FROM read_parquet('{test_pattern}')
)
TO 'data/processed/CA_1_features_v1.parquet'
(FORMAT PARQUET, COMPRESSION ZSTD);
""")

In [13]:
con.execute("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT item_id) AS items,
    COUNT(DISTINCT date) AS dates
FROM read_parquet('data/processed/CA_1_features_v1.parquet')
""").fetchdf()

,rows,items,dates
0,5918109,3049,1941


In [14]:
con.execute("""
SELECT
    *
FROM read_parquet('data/processed/CA_1_features_v1.parquet')
LIMIT 10
""").fetchdf()

,item_id,dept_id,cat_id,store_id,state_id,d,units_sold,date,wm_yr_wk,weekday,...,event_type_1,event_name_2,event_type_2,sell_price,is_available,lag_1,lag_7,lag_28,rolling_mean_7,rolling_mean_28
0,FOODS_2_083,FOODS_2,FOODS,CA_1,CA,d_750,0,2013-02-16,11304,Saturday,...,None,None,None,6.98,1,3,4,1,2.428571,1.392857
1,FOODS_2_083,FOODS_2,FOODS,CA_1,CA,d_751,0,2013-02-17,11304,Sunday,...,None,None,None,6.98,1,0,3,0,1.857143,1.357143
2,FOODS_2_083,FOODS_2,FOODS,CA_1,CA,d_752,0,2013-02-18,11304,Monday,...,National,None,None,6.98,1,0,1,1,1.428571,1.357143
3,FOODS_2_083,FOODS_2,FOODS,CA_1,CA,d_753,0,2013-02-19,11304,Tuesday,...,None,None,None,6.98,1,0,2,1,1.285714,1.321429
4,FOODS_2_083,FOODS_2,FOODS,CA_1,CA,d_754,1,2013-02-20,11304,Wednesday,...,Religious,None,None,6.98,1,0,1,0,1.000000,1.285714
5,FOODS_2_083,FOODS_2,FOODS,CA_1,CA,d_755,0,2013-02-21,11304,Thursday,...,None,None,None,6.98,1,1,3,0,1.000000,1.321429
6,FOODS_2_083,FOODS_2,FOODS,CA_1,CA,d_756,1,2013-02-22,11304,Friday,...,None,None,None,6.98,1,0,3,0,0.571429,1.321429
7,FOODS_2_083,FOODS_2,FOODS,CA_1,CA,d_757,1,2013-02-23,11305,Saturday,...,None,None,None,6.98,1,1,0,0,0.285714,1.357143
8,FOODS_2_083,FOODS_2,FOODS,CA_1,CA,d_758,1,2013-02-24,11305,Sunday,...,Religious,None,None,6.98,1,1,0,2,0.428571,1.392857
9,FOODS_2_083,FOODS_2,FOODS,CA_1,CA,d_759,0,2013-02-25,11305,Monday,...,None,None,None,6.98,1,1,0,1,0.571429,1.357143
